In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [2]:
df = pd.read_csv('../../datasets/c_text_youm7/youm7_nlp_preprocessed_data.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12089 entries, 0 to 12088
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   category_label   12089 non-null  str  
 1   summary_no_stem  12089 non-null  str  
 2   summary_stemmed  12089 non-null  str  
dtypes: str(3)
memory usage: 5.2 MB


# Building Logistic Regression

In [4]:
class LogisticRegressionScratch(ClassifierMixin, BaseEstimator):
    def __init__(self, learning_rate=5.0, iterations=500, C=10.0, momentum=0.9):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.C = C
        self.momentum = momentum # keeps part of previous step to speed up training (accelerate convergence)
        
    def sigmoid(self, z):
        z = np.clip(z, -500, 500) # clip to avoid overflow in exp
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y) 
        m, n = X.shape # number of samples and features
        self.weights = {} # dictionary to store weights for each class
        self.biases = {} # dictionary to store biases for each class

        for cls in self.classes_: # one-vs-rest approach (OVR)
            y_binary = (y == cls).astype(float) # convert to binary labels for the current class
            w = np.zeros(n) # initialize weights for the current class
            b = 0.0 # initialize bias for the current class
            vw = np.zeros(n) # initialize velocity for weights (momentum)
            vb = 0.0 # initialize velocity for bias (momentum)

            for _ in range(self.iterations): # gradient descent loop
                z = (X @ w) + b
                h = self.sigmoid(z)
                dw = (1 / m) * (X.T @ (h - y_binary)) + (1 / (self.C * m)) * w # weight gradient with L2 regularization
                db = (1 / m) * np.sum(h - y_binary) # bias gradient (no regularization on bias)
                vw = self.momentum * vw + dw 
                vb = self.momentum * vb + db 
                w -= self.learning_rate * vw
                b -= self.learning_rate * vb

            self.weights[cls] = w
            self.biases[cls] = b

        return self

    def decision_function(self, X): # returns the raw scores for each class
        scores = []
        for cls in self.classes_:
            z = (X @ self.weights[cls]) + self.biases[cls] # compute the linear combination of inputs and weights for the current class
            scores.append(z)
        return np.column_stack(scores)

    def predict(self, X): # returns the predicted class labels for the input samples
        scores = self.decision_function(X) 
        best_class_index = np.argmax(scores, axis=1) # get the index of the class with the highest score for each sample
        return self.classes_[best_class_index]

# Stemmed Training

In [5]:
df_stemmed = df[['category_label', 'summary_stemmed']].copy()

In [6]:
# 70% Train, 30% Temp (Dev + Test)
X_train_stem, X_temp_stem, y_train_stem, y_temp_stem = train_test_split(
    df_stemmed["summary_stemmed"],
    df_stemmed["category_label"],
    test_size=0.30,
    random_state=42,
    stratify=df_stemmed["category_label"]
)

# 15% Dev, 15% Test
X_dev_stem, X_test_stem, y_dev_stem, y_test_stem = train_test_split(
    X_temp_stem,
    y_temp_stem,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_stem
)

In [7]:
C_values = [0.001, 0.01, 0.1, 1, 2, 5, 10, 20, 50, 100]

best_c_stem = None
best_dev_f1_stem = -1

results_stem = []

for c in C_values:
    candidate = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            norm="l2",
            sublinear_tf=True
        ),
        LogisticRegressionScratch(
            C=c,
            learning_rate=5.0,
            iterations=1000,
            momentum=0.9
        )
    )

    candidate.fit(X_train_stem, y_train_stem)

    # Train evaluation
    train_preds = candidate.predict(X_train_stem)
    train_f1 = f1_score(
        y_train_stem,
        train_preds,
        average="macro"
    )

    # Dev evaluation
    dev_preds = candidate.predict(X_dev_stem)
    dev_f1 = f1_score(
        y_dev_stem,
        dev_preds,
        average="macro"
    )
    dev_acc = accuracy_score(y_dev_stem, dev_preds)

    results_stem.append({
        "C": c,
        "Train Macro F1": train_f1,
        "Dev Macro F1": dev_f1,
        "Dev Accuracy": dev_acc
    })

    if dev_f1 > best_dev_f1_stem:
        best_dev_f1_stem = dev_f1
        best_c_stem = c

results_stem_df = pd.DataFrame(results_stem)

print(results_stem_df.to_string(index=False))
print(f"\nBest C (Stemmed): {best_c_stem}")
print(f"Best Dev Macro F1 (Stemmed): {best_dev_f1_stem:.4f}")

      C  Train Macro F1  Dev Macro F1  Dev Accuracy
  0.001        0.038428      0.038459      0.155543
  0.010        0.625137      0.672112      0.670712
  0.100        0.892736      0.876902      0.876999
  1.000        0.964547      0.912231      0.912300
  2.000        0.980283      0.916108      0.916161
  5.000        0.993691      0.918845      0.918919
 10.000        0.996274      0.919426      0.919470
 20.000        0.997100      0.919948      0.920022
 50.000        0.997473      0.919414      0.919470
100.000        0.997473      0.919414      0.919470

Best C (Stemmed): 20
Best Dev Macro F1 (Stemmed): 0.9199


In [8]:
X_train_final_stem = pd.concat([X_train_stem, X_dev_stem])
y_train_final_stem = pd.concat([y_train_stem, y_dev_stem])

In [9]:
lr_stemmed_model = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        norm="l2",
        sublinear_tf=True
    ),
    LogisticRegressionScratch(
        C=best_c_stem,
        learning_rate=5.0,
        iterations=1000,
        momentum=0.9
    )
)

lr_stemmed_model.fit(
    X_train_final_stem,
    y_train_final_stem
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregressionscratch', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['اقتصاد_وبورصة','حوادث_وقضايا','رياضة_مصرية',...,'علوم_وتكنولوجيا', 'فن_وثقافة','محافظات_ومجتمع']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [10]:
test_preds_stem = lr_stemmed_model.predict(X_test_stem)

print(f"Test Accuracy: {accuracy_score(y_test_stem, test_preds_stem):.4f}")
print(f"Test Macro F1: {f1_score(y_test_stem, test_preds_stem, average='macro'):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_stem,
        test_preds_stem,
        zero_division=0
    )
)

Test Accuracy: 0.9272
Test Macro F1: 0.9274

Classification Report:
                  precision    recall  f1-score   support

   اقتصاد_وبورصة       0.93      0.93      0.93       261
    حوادث_وقضايا       0.92      0.95      0.93       223
     رياضة_مصرية       0.98      0.98      0.98       252
سياسة_وأخبار_مصر       0.93      0.85      0.89       268
 علوم_وتكنولوجيا       0.91      0.96      0.94       282
       فن_وثقافة       0.93      0.91      0.92       262
  محافظات_ومجتمع       0.90      0.91      0.91       266

        accuracy                           0.93      1814
       macro avg       0.93      0.93      0.93      1814
    weighted avg       0.93      0.93      0.93      1814



# Testing valid across both stemming, non stemmed

In [11]:
def get_classification_results(
    y_true,
    y_pred,
    decision_scores,
    classes
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    decision_scores = np.asarray(decision_scores)

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        labels=classes,
        output_dict=True,
        zero_division=0
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=classes
    )

    rows = []

    # Per-class metrics
    for i, label in enumerate(classes):

        metrics = report[str(label)]

        # One-vs-Rest binary classification
        y_binary = (y_true == label).astype(int)

        # True Positives, False Positives,
        # True Negatives, False Negatives
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        # Specificity
        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0 else np.nan
        )

        # ROC-AUC and PR-AUC
        if len(np.unique(y_binary)) == 2:

            roc_auc = roc_auc_score(
                y_binary,
                decision_scores[:, i]
            )

            pr_auc = average_precision_score(
                y_binary,
                decision_scores[:, i]
            )

        else:
            roc_auc = np.nan
            pr_auc = np.nan

        rows.append({
            "Category": label,
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1-score": metrics["f1-score"],
            "Specificity": specificity,
            "ROC-AUC": roc_auc,
            "PR-AUC (AP)": pr_auc,
            "Accuracy": np.nan,
            "Support": int(metrics["support"])
        })

    # Create per-class DataFrame
    results_df = pd.DataFrame(rows)

    # Macro Average
    macro_row = {
        "Category": "Macro Average",
        "Precision": report["macro avg"]["precision"],
        "Recall": report["macro avg"]["recall"],
        "F1-score": report["macro avg"]["f1-score"],
        "Specificity": results_df["Specificity"].mean(),
        "ROC-AUC": results_df["ROC-AUC"].mean(),
        "PR-AUC (AP)": results_df["PR-AUC (AP)"].mean(),
        "Accuracy": np.nan,
        "Support": len(y_true)
    }

    # Weighted Average
    weights = results_df["Support"].to_numpy()

    weighted_row = {
        "Category": "Weighted Average",
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1-score": report["weighted avg"]["f1-score"],
        "Specificity": np.average(
            results_df["Specificity"],
            weights=weights
        ),
        "ROC-AUC": np.average(
            results_df["ROC-AUC"],
            weights=weights
        ),
        "PR-AUC (AP)": np.average(
            results_df["PR-AUC (AP)"],
            weights=weights
        ),
        "Accuracy": np.nan,
        "Support": len(y_true)
    }

    # Overall Accuracy
    accuracy_row = {
        "Category": "Overall Accuracy",
        "Precision": np.nan,
        "Recall": np.nan,
        "F1-score": np.nan,
        "Specificity": np.nan,
        "ROC-AUC": np.nan,
        "PR-AUC (AP)": np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Support": len(y_true)
    }

    # Combine all rows
    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([
                macro_row,
                weighted_row,
                accuracy_row
            ])
        ],
        ignore_index=True
    )

    return results_df

# Testing the Stemmed Logistic Regression

In [12]:
lr_stemmed_test_preds = lr_stemmed_model.predict(X_test_stem)

lr_stemmed_test_scores = (
    lr_stemmed_model.decision_function(X_test_stem)
)

lr_stemmed_results = get_classification_results(
    y_true=y_test_stem,
    y_pred=lr_stemmed_test_preds,
    decision_scores=lr_stemmed_test_scores,
    classes=lr_stemmed_model.classes_
)

In [13]:
display(
    lr_stemmed_results.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "Specificity": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC (AP)": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Support": "{:,.0f}"
    }, na_rep="—")
)

,Category,Precision,Recall,F1-score,Specificity,ROC-AUC,PR-AUC (AP),Accuracy,Support
0,اقتصاد_وبورصة,0.9313,0.9349,0.9331,0.9884,0.9936,0.9751,—,261
1,حوادث_وقضايا,0.9174,0.9462,0.9316,0.9881,0.9966,0.9737,—,223
2,رياضة_مصرية,0.9801,0.9762,0.9781,0.9968,0.9990,0.9950,—,252
3,سياسة_وأخبار_مصر,0.9309,0.8545,0.8911,0.9890,0.9841,0.9470,—,268
4,علوم_وتكنولوجيا,0.9125,0.9610,0.9361,0.9830,0.9960,0.9761,—,282
5,فن_وثقافة,0.9261,0.9084,0.9171,0.9878,0.9912,0.9660,—,262
6,محافظات_ومجتمع,0.8967,0.9135,0.9050,0.9819,0.9904,0.9448,—,266
7,Macro Average,0.9278,0.9278,0.9274,0.9879,0.9930,0.9682,—,"1,814"
8,Weighted Average,0.9275,0.9272,0.9270,0.9877,0.9929,0.9679,—,"1,814"
9,Overall Accuracy,—,—,—,—,—,—,0.9272,"1,814"


# No Stem Training

In [14]:
df_no_stem = df[['category_label', 'summary_no_stem']].copy() # DataFrame for no-stem results

In [15]:
# 70% Train, 30% Temp (Dev + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df_no_stem["summary_no_stem"],
    df_no_stem["category_label"],
    test_size=0.30,
    random_state=42,
    stratify=df_no_stem["category_label"]
)

# 15% Dev, 15% Test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [16]:
C_values = [0.001, 0.01, 0.1, 1, 2, 5, 10, 20, 50, 100] 

best_c = None
best_dev_f1 = -1 

results = []

for c in C_values:

    candidate = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            norm="l2",
            sublinear_tf=True
        ),

        LogisticRegressionScratch(
            C=c,
            learning_rate=5.0,
            iterations=1000,
            momentum=0.9
        )
    )

    candidate.fit(X_train, y_train)

    # Train evaluation
    train_preds = candidate.predict(X_train)

    train_f1 = f1_score(
        y_train,
        train_preds,
        average="macro"
    )

    # Dev evaluation
    dev_preds = candidate.predict(X_dev)

    dev_f1 = f1_score(
        y_dev,
        dev_preds,
        average="macro"
    )

    dev_acc = accuracy_score(y_dev, dev_preds)

    results.append({
        "C": c,
        "Train Macro F1": train_f1,
        "Dev Macro F1": dev_f1,
        "Dev Accuracy": dev_acc
    })

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_c = c


results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

print(f"\nBest C: {best_c}")
print(f"Best Dev Macro F1: {best_dev_f1:.4f}")

      C  Train Macro F1  Dev Macro F1  Dev Accuracy
  0.001        0.038428      0.038459      0.155543
  0.010        0.517891      0.571475      0.557639
  0.100        0.897653      0.862471      0.861004
  1.000        0.975305      0.900866      0.900717
  2.000        0.988658      0.910868      0.910645
  5.000        0.996302      0.916365      0.916161
 10.000        0.997828      0.917556      0.917264
 20.000        0.998315      0.918594      0.918367
 50.000        0.998441      0.917993      0.917816
100.000        0.998554      0.917993      0.917816

Best C: 20
Best Dev Macro F1: 0.9186


In [17]:
X_train_final = pd.concat([X_train, X_dev])
y_train_final = pd.concat([y_train, y_dev])

In [18]:
lr_no_stem_model = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        norm="l2",
        sublinear_tf=True
    ),
    LogisticRegressionScratch(
        C=best_c,
        learning_rate=5.0,
        iterations=1000,
        momentum=0.9
    )
)

In [19]:
lr_no_stem_model.fit(
    X_train_final,
    y_train_final
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregressionscratch', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['اقتصاد_وبورصة','حوادث_وقضايا','رياضة_مصرية',...,'علوم_وتكنولوجيا', 'فن_وثقافة','محافظات_ومجتمع']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [20]:
test_preds = lr_no_stem_model.predict(X_test)

print(
    f"Test Accuracy: "
    f"{accuracy_score(y_test, test_preds):.4f}"
)

print(
    f"Test Macro F1: "
    f"{f1_score(y_test, test_preds, average='macro'):.4f}"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_preds,
        zero_division=0
    )
)

Test Accuracy: 0.9361
Test Macro F1: 0.9363

Classification Report:
                  precision    recall  f1-score   support

   اقتصاد_وبورصة       0.95      0.93      0.94       261
    حوادث_وقضايا       0.93      0.95      0.94       223
     رياضة_مصرية       0.99      0.99      0.99       252
سياسة_وأخبار_مصر       0.95      0.87      0.91       268
 علوم_وتكنولوجيا       0.91      0.98      0.94       282
       فن_وثقافة       0.94      0.92      0.93       262
  محافظات_ومجتمع       0.90      0.92      0.91       266

        accuracy                           0.94      1814
       macro avg       0.94      0.94      0.94      1814
    weighted avg       0.94      0.94      0.94      1814



# Testing No Stem Logistic Regression

In [21]:
lr_no_stem_test_preds = lr_no_stem_model.predict(X_test)

lr_no_stem_test_scores = (
    lr_no_stem_model.decision_function(X_test)
)

lr_no_stem_results = get_classification_results(
    y_true=y_test,
    y_pred=lr_no_stem_test_preds,
    decision_scores=lr_no_stem_test_scores,
    classes=lr_no_stem_model.classes_
)

In [22]:
display(
    lr_no_stem_results.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "Specificity": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC (AP)": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Support": "{:,.0f}"
    }, na_rep="—")
)

,Category,Precision,Recall,F1-score,Specificity,ROC-AUC,PR-AUC (AP),Accuracy,Support
0,اقتصاد_وبورصة,0.9457,0.9349,0.9403,0.9910,0.9955,0.9798,—,261
1,حوادث_وقضايا,0.9254,0.9462,0.9357,0.9893,0.9964,0.9709,—,223
2,رياضة_مصرية,0.9881,0.9881,0.9881,0.9981,0.9994,0.9961,—,252
3,سياسة_وأخبار_مصر,0.9547,0.8657,0.9080,0.9929,0.9839,0.9435,—,268
4,علوم_وتكنولوجيا,0.9076,0.9752,0.9402,0.9817,0.9933,0.9654,—,282
5,فن_وثقافة,0.9414,0.9198,0.9305,0.9903,0.9924,0.9741,—,262
6,محافظات_ومجتمع,0.8978,0.9248,0.9111,0.9819,0.9907,0.9444,—,266
7,Macro Average,0.9373,0.9364,0.9363,0.9893,0.9931,0.9677,—,"1,814"
8,Weighted Average,0.9369,0.9361,0.9359,0.9892,0.9929,0.9673,—,"1,814"
9,Overall Accuracy,—,—,—,—,—,—,0.9361,"1,814"
